# 05. Streaming: many metrics, guardrails, and multiple-testing correction

**Sector:** video streaming. **Decision:** launch the new recommendation
algorithm? The problem is not one metric, it is **several**: watch time
(primary), number of sessions, completion rate, and a buffering **guardrail**
(must not get worse). Testing many metrics inflates the chance of a false
positive, so we must **correct for multiple testing**.

One experiment, one allocation, many metrics. We close with an
`ExperimentPipeline` and an `ExperimentReport`. Theory:
[IV. Inference](../../../docs/en/theory/04-inference.md) (topic 16) and
[V. Diagnostics](../../../docs/en/theory/05-diagnostics.md) (topic 21).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "streaming_metrics.csv")
print(df.shape)
df.head()


## 1. One allocation, several metrics

We randomize users **once** and reuse the **same** allocation for every metric
(it is the same experiment). For each metric, we build the observed outcome and
estimate the effect with `NeymanCI`.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI

design = CRD(p=0.5, seed=505)
base = design.randomize(df[["plan"]].copy())
t = base.data_[base.treatment_col_].to_numpy()

metrics = ["watch_time", "sessions", "completion", "buffering"]
results = {}
for m in metrics:
    data = base.data_.copy()
    data[m] = np.where(t == 1, df[f"{m}_y1"].to_numpy(), df[f"{m}_y0"].to_numpy())
    a = CRDAssignment(data=data, treatment_col=base.treatment_col_, design=design, seed=505)
    results[m] = NeymanCI(estimator=DifferenceInMeans(outcome_col=m)).fit(a).estimate()

{m: round(r.p_value, 4) for m, r in results.items()}


## 2. Multiple-testing correction (FWER, Holm)

With four metrics, we control the family-wise error rate with `Holm`.
`ExperimentComparison` applies the correction over the family and returns a
ready-made table, plus the forest plot.


In [ ]:
from skxperiments.pipeline import ExperimentComparison
from skxperiments.reporting import plot_forest

comparison = ExperimentComparison(correction="holm", alpha=0.05).run(results)
display_cols = ["experiment", "ate", "p_value", "p_value_corrected", "significant"]
print(comparison.table[display_cols].round(4).to_string(index=False))

ax = plot_forest(comparison)
ax.set_title("Effect by metric (Holm-corrected)")
ax.figure


## 3. Pipeline and report on the primary metric

Only **watch time** survives the correction; sessions and completion are not
significant, and the buffering guardrail is **not flagged** (no harm). We close
with the `ExperimentPipeline` on the primary metric, which runs the `SRMTest`
automatically, and generate an HTML `ExperimentReport`.


In [ ]:
from skxperiments.diagnostics import SRMTest
from skxperiments.pipeline import ExperimentPipeline

data = base.data_.copy()
data["watch_time"] = np.where(t == 1, df["watch_time_y1"].to_numpy(), df["watch_time_y0"].to_numpy())
primary = CRDAssignment(data=data, treatment_col=base.treatment_col_, design=design, seed=505)

pipeline_result = ExperimentPipeline(
    inference=NeymanCI(estimator=DifferenceInMeans(outcome_col="watch_time")),
    diagnostics=[SRMTest()],
).run(primary)
print(f"watch_time ATE={pipeline_result.results.ate:.3f}, "
      f"p={pipeline_result.results.p_value:.4f}, flagged={pipeline_result.flagged}")


In [ ]:
from IPython.display import HTML

from skxperiments.reporting import ExperimentReport

report = ExperimentReport(pipeline_result, title="Streaming recommender - watch time")
HTML(report.to_html())


## Decision

The true effect on watch time is `+3` minutes, and it is the only real one; the
other metrics were generated with no effect (or a tiny one). The Holm correction
does exactly the right job: it keeps the primary discovery and does not let the
noise of the other metrics turn into a "win". With the guardrail clean, the
recommendation is to **launch**, monitoring buffering.

This closes the use-cases track. For the fundamentals behind each piece, see the
[theory series](../../../docs/en/theory/01-foundations.md).
